# 06 — Reproducibility & Provenance

**Purpose:** Explain and demonstrate how PBI tracks the identity and provenance of a
data release, so that results can be reproduced and cited correctly.

| | |
|---|---|
| **Expected inputs** | Pipeline logs at `/pipeline-logs` (optional but recommended), `pbi` package installed |
| **Outputs** | Provenance summary saved under `<results>/06_reproducibility/` |

## What you will learn

1. The **three-layer versioning model** used in this project
2. How to read the **pbi package version** programmatically
3. How the **provider schema profile** (`phagescope_v1`) is defined and when it changes
4. How to read and validate **pipeline run provenance** artefacts
5. How to verify the **public data manifest** (snapshot date, checksums)
6. How to inspect **database build metadata**
7. A practical **reproducibility checklist**


---
## Setup


In [1]:
import sys
import json
import os
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

import pbi
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

logs_root = Path('/pipeline-logs')

results_root = Path(os.getenv('PBI_RESULTS_DIR', '/results'))
results_dir = results_root / '06_reproducibility'
results_dir.mkdir(parents=True, exist_ok=True)

print(f'pbi version  : {pbi.__version__}')
print(f'Python       : {sys.version.split()[0]}')
print(f'Results dir  : {results_dir}')


pbi version  : 0.3.0
Python       : 3.10.20
Results dir  : /results/06_reproducibility


---
## 1. The Three-Layer Versioning Model

PBI deliberately separates three concepts that are often conflated:

```
┌─────────────────────────────────────────────────────────────────────┐
│  Layer 1 — Package version                                          │
│  ─────────────────────────────────────────────────────────────────  │
│  SemVer string in src/pbi/__init__.py   e.g. "0.3.0"               │
│  Read by setup.py; tracked with Git tags (v0.3.0).                  │
│  Bump when the pbi Python API changes.                              │
│                                                                     │
│  Layer 2 — Provider schema profile                                  │
│  ─────────────────────────────────────────────────────────────────  │
│  Stable label in workflow/config/config.yaml                        │
│  public_data_provider.schema_profile   e.g. "phagescope_v1"        │
│  Records the column/semantic contract of the upstream data source.  │
│  Bump to "phagescope_v2" only when the source schema changes.       │
│                                                                     │
│  Layer 3 — Pipeline run provenance                                  │
│  ─────────────────────────────────────────────────────────────────  │
│  JSON artefact written per pipeline execution                       │
│  /pipeline-logs/csv/pipeline_run_provenance.json                    │
│  Records: snapshot_date, git_ref, git_commit, pbi_version, ...      │
└─────────────────────────────────────────────────────────────────────┘
```

**Key principle:** keeping them separate means you can compare two pipeline runs
that used the same schema profile (same data contract) even if the package version
changed between runs.  Conversely, a package update that does not change data
semantics does not force a schema version bump.


---
## 2. Layer 1 — Package Version


In [2]:
# The single source of truth for the package version is __version__ in
# src/pbi/__init__.py.  setup.py reads it with a regex, so no import is
# needed at build time.

print(f'pbi.__version__ = {repr(pbi.__version__)}')

# Cross-check: read the file line-by-line (avoids loading the full file)
import re
init_py = project_root / 'src' / 'pbi' / '__init__.py'
if init_py.exists():
    file_version = 'not found'
    with init_py.open() as _fh:
        for _line in _fh:
            _m = re.match(r"^__version__\s*=\s*[\"'](.*?)[\"']", _line)
            if _m:
                file_version = _m.group(1)
                break
    match_ok = file_version == pbi.__version__
    print(f'Version in file : {repr(file_version)}  {"✅ match" if match_ok else "❌ mismatch"}')


pbi.__version__ = '0.3.0'


---
## 3. Layer 2 — Provider Schema Profile

The `schema_profile` field lives in `workflow/config/config.yaml` under
`public_data_provider`.  It is written into every download manifest and into the
DuckDB `dataset_provenance` table during the pipeline run, so it is always
traceable from the data artefacts alone.

Changing this label is a **deliberate, documented decision** — it signals to
downstream consumers that the column layout or data semantics changed.


In [3]:
# Try to read schema_profile from the workflow config (repo checkout only)
try:
    import yaml  # PyYAML is a pbi dependency
    config_path = project_root / 'workflow' / 'config' / 'config.yaml'
    if config_path.exists():
        with config_path.open() as fh:
            cfg = yaml.safe_load(fh)
        provider = cfg.get('public_data_provider', {})
        print('public_data_provider (from workflow/config/config.yaml):')
        for k, v in provider.items():
            print(f'  {k:<30} {v}')
    else:
        print('workflow/config/config.yaml not found (release package only — see manifest).')
except ImportError:
    print('PyYAML not available; install pyyaml or inspect config.yaml manually.')


workflow/config/config.yaml not found (release package only — see manifest).


---
## 4. Layer 3 — Pipeline Run Provenance

The pipeline writes `/pipeline-logs/csv/pipeline_run_provenance.json` at the
end of a successful run.  Reading it confirms *exactly* which data snapshot,
software version, and (optionally) git commit produced the current database.


In [4]:
prov_path = logs_root / 'csv' / 'pipeline_run_provenance.json'

if prov_path.exists():
    provenance = json.loads(prov_path.read_text())
    print('pipeline_run_provenance.json')
    print('─' * 55)
    for key, val in provenance.items():
        print(f'  {key:<40} {val}')

    # Save a copy to results for archiving
    out = results_dir / 'pipeline_run_provenance.json'
    out.write_text(json.dumps(provenance, indent=2))
    print(f'\nCopy saved → {out}')
else:
    provenance = {}
    print('ℹ️  pipeline_run_provenance.json not found.')
    print('   Run the Snakemake pipeline to generate this artefact.')


pipeline_run_provenance.json
───────────────────────────────────────────────────────
  pipeline_run_timestamp                   2026-07-04T15:56:50Z
  provider_name                            PhageScope
  provider_release                         rolling
  provider_snapshot_date                   2026-05-11
  provider_schema_profile                  phagescope_v1
  provider_api_base_url                    https://phageapi.deepomics.org
  provider_provenance_mode                 config_pinned
  pbi_version                              
  git_commit                               
  download_records_count                   229

Copy saved → /results/06_reproducibility/pipeline_run_provenance.json


---
## 5. Public Data Manifest

The pipeline also writes `/pipeline-logs/csv/public_data_manifest.json` containing
one entry per downloaded file: URL, HTTP response headers (including `Last-Modified`
and `ETag`), file size, and optionally a checksum.  Together with the provenance
record this allows exact reconstruction of the input data.


In [5]:
manifest_path = logs_root / 'csv' / 'public_data_manifest.json'

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    # Manifest may be a list or a dict — handle both
    entries = manifest if isinstance(manifest, list) else manifest.get('files', [])
    print(f'Public data manifest — {len(entries)} entries')
    if entries:
        df_manifest = pd.json_normalize(entries[:5])  # preview first 5
        display(df_manifest)
else:
    print('ℹ️  public_data_manifest.json not found.')
    print('   Generated by the download_public_file pipeline step.')


Public data manifest — 229 entries


,provider_name,provider_release,provider_snapshot_date,provider_schema_profile,feature,source_key,normalized_source_db,source_url,local_path,retrieved_at,file_size,sha256,etag,last_modified,detected_tabular_columns,schema_fingerprint,status,error_message
0,PhageScope,rolling,2026-05-11,phagescope_v1,annotated_proteins_metadata,BAPS_Annotated_Proteins_metadata_URL,BAPS,https://phageapi.deepomics.org/files/Download/Annotated_protein_meta_data_v2...,/data/intermediate/csv/annotated_proteins_metadata/BAPS_Annotated_Proteins_m...,2026-07-04T15:43:24Z,351148257,0f46611cde9d0b39feac261829e158cad52a7ec1616c0db1c963bcda44a656a6,None,None,"[Phage_ID, Protein_source, Function_Prediction_source, Start, Stop, Strand, ...",8c2f7b24d74f40a3,success,
1,PhageScope,rolling,2026-05-11,phagescope_v1,annotated_proteins_metadata,CHVD_Annotated_Proteins_metadata_URL,CHVD,https://phageapi.deepomics.org/files/Download/Annotated_protein_meta_data_v2...,/data/intermediate/csv/annotated_proteins_metadata/CHVD_Annotated_Proteins_m...,2026-07-04T15:39:15Z,477743473,1450b0a62a6c808d5fce487620f3792459509a64e3ed19f48f72896220495369,None,None,"[Phage_ID, Protein_source, Function_Prediction_source, Start, Stop, Strand, ...",8c2f7b24d74f40a3,success,
2,PhageScope,rolling,2026-05-11,phagescope_v1,annotated_proteins_metadata,DDBJ_Annotated_Proteins_metadata_URL,DDBJ,https://phageapi.deepomics.org/files/Download/Annotated_protein_meta_data_v2...,/data/intermediate/csv/annotated_proteins_metadata/DDBJ_Annotated_Proteins_m...,2026-07-04T15:48:30Z,3602250,5bc08733991bd3e97f34a012169de45cb7af84bbbeb3f9c50457ac2a809a71eb,None,None,"[Phage_ID, Protein_source, Function_prediction_source, Start, Stop, Strand, ...",9d5b32b8ebd8d1ae,success,
3,PhageScope,rolling,2026-05-11,phagescope_v1,annotated_proteins_metadata,ELGV_Annotated_Proteins_metadata_URL,ELGV,https://phageapi.deepomics.org/files/Download/Annotated_protein_meta_data_v2...,/data/intermediate/csv/annotated_proteins_metadata/ELGV_Annotated_Proteins_m...,2026-07-04T15:42:37Z,342584275,5b68c692707b25d923b54710d6ce19c49e2e3aaea42e72d5c3e3fc23e834fa44,None,None,"[Phage_ID, Protein_source, Function_Prediction_source, Start, Stop, Strand, ...",8c2f7b24d74f40a3,success,
4,PhageScope,rolling,2026-05-11,phagescope_v1,annotated_proteins_metadata,EMBL_Annotated_Proteins_metadata_URL,EMBL,https://phageapi.deepomics.org/files/Download/Annotated_protein_meta_data_v2...,/data/intermediate/csv/annotated_proteins_metadata/EMBL_Annotated_Proteins_m...,2026-07-04T15:45:12Z,2307398,5bd93b9a30c84da42d8128f3ae5bdd4a864c383514d120b949e5c22ca4bc495d,None,None,"[Phage_ID, Protein_source, Function_prediction_source, Start, Stop, Strand, ...",9d5b32b8ebd8d1ae,success,


---
## 6. Database Build Metadata

The DuckDB database contains a `dataset_provenance` table (and optionally a
`run_provenance` table) that record the same metadata directly inside the
artefact.  This means the provenance travels with the database file, even when
it is distributed via Zenodo.


In [6]:
try:
    retriever = pbi.quick_connect()
    conn = retriever.conn

    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchdf()
    prov_tables = [t for t in tables['name'].tolist()
                   if 'provenance' in t.lower() or 'metadata' in t.lower()]

    if prov_tables:
        for tbl in prov_tables:
            print(f'\n── {tbl} ──')
            try:
                display(conn.execute(f'SELECT * FROM {tbl} LIMIT 5').fetchdf())
            except Exception as exc:
                print(f'  Could not query {tbl}: {exc}')
    else:
        print('No provenance tables found in database.')
        print('Available tables:', tables['name'].tolist())

    retriever.close()
except Exception as exc:
    print(f'Could not connect to database: {exc}')
    print('Run the pipeline first or check DATA_PATH.')


2026-07-06 06:39:46,973 - INFO - 📂 Checking FASTA index files:
2026-07-06 06:39:46,974 - INFO -    Phage index: True (82826.5 KB)
2026-07-06 06:39:46,975 - INFO -    Protein index: True (4376442.2 KB)
2026-07-06 06:39:46,977 - INFO - 📂 Loaded private phage mapping for 2 sources: ['test_private', 'test_private_2']
2026-07-06 06:39:46,977 - INFO - 📂 Using host mapping file: /data/processed/sequences/host_fasta_mapping.json
2026-07-06 06:39:46,982 - INFO -    Loaded mapping for 5517 hosts
2026-07-06 06:39:46,983 - INFO - 📂 Connecting to database: /data/processed/databases/phage_database_optimized.duckdb
2026-07-06 06:39:47,019 - INFO - 🔄 Starting background FASTA loading...
2026-07-06 06:39:47,020 - INFO - 🔄 [Background] Loading phage FASTA: /data/processed/sequences/all_phages.fasta
2026-07-06 06:39:47,021 - INFO - ✅ Initialization complete (FASTA loading in background)



── dataset_provenance ──


,provider_name,provider_release,provider_snapshot_date,provider_schema_profile,feature,source_key,normalized_source_db,source_url,local_path,retrieved_at,file_size,sha256,etag,last_modified,detected_tabular_columns,schema_fingerprint,status,error_message
0,PhageScope,rolling,2026-05-11,phagescope_v1,annotated_proteins_metadata,BAPS_Annotated_Proteins_metadata_URL,BAPS,https://phageapi.deepomics.org/files/Download/Annotated_protein_meta_data_v2...,/data/intermediate/csv/annotated_proteins_metadata/BAPS_Annotated_Proteins_m...,2026-07-04T15:43:24Z,351148257,0f46611cde9d0b39feac261829e158cad52a7ec1616c0db1c963bcda44a656a6,None,None,"[""Phage_ID"", ""Protein_source"", ""Function_Prediction_source"", ""Start"", ""Stop""...",8c2f7b24d74f40a3,success,None
1,PhageScope,rolling,2026-05-11,phagescope_v1,annotated_proteins_metadata,CHVD_Annotated_Proteins_metadata_URL,CHVD,https://phageapi.deepomics.org/files/Download/Annotated_protein_meta_data_v2...,/data/intermediate/csv/annotated_proteins_metadata/CHVD_Annotated_Proteins_m...,2026-07-04T15:39:15Z,477743473,1450b0a62a6c808d5fce487620f3792459509a64e3ed19f48f72896220495369,None,None,"[""Phage_ID"", ""Protein_source"", ""Function_Prediction_source"", ""Start"", ""Stop""...",8c2f7b24d74f40a3,success,None
2,PhageScope,rolling,2026-05-11,phagescope_v1,annotated_proteins_metadata,DDBJ_Annotated_Proteins_metadata_URL,DDBJ,https://phageapi.deepomics.org/files/Download/Annotated_protein_meta_data_v2...,/data/intermediate/csv/annotated_proteins_metadata/DDBJ_Annotated_Proteins_m...,2026-07-04T15:48:30Z,3602250,5bc08733991bd3e97f34a012169de45cb7af84bbbeb3f9c50457ac2a809a71eb,None,None,"[""Phage_ID"", ""Protein_source"", ""Function_prediction_source"", ""Start"", ""Stop""...",9d5b32b8ebd8d1ae,success,None
3,PhageScope,rolling,2026-05-11,phagescope_v1,annotated_proteins_metadata,ELGV_Annotated_Proteins_metadata_URL,ELGV,https://phageapi.deepomics.org/files/Download/Annotated_protein_meta_data_v2...,/data/intermediate/csv/annotated_proteins_metadata/ELGV_Annotated_Proteins_m...,2026-07-04T15:42:37Z,342584275,5b68c692707b25d923b54710d6ce19c49e2e3aaea42e72d5c3e3fc23e834fa44,None,None,"[""Phage_ID"", ""Protein_source"", ""Function_Prediction_source"", ""Start"", ""Stop""...",8c2f7b24d74f40a3,success,None
4,PhageScope,rolling,2026-05-11,phagescope_v1,annotated_proteins_metadata,EMBL_Annotated_Proteins_metadata_URL,EMBL,https://phageapi.deepomics.org/files/Download/Annotated_protein_meta_data_v2...,/data/intermediate/csv/annotated_proteins_metadata/EMBL_Annotated_Proteins_m...,2026-07-04T15:45:12Z,2307398,5bd93b9a30c84da42d8128f3ae5bdd4a864c383514d120b949e5c22ca4bc495d,None,None,"[""Phage_ID"", ""Protein_source"", ""Function_prediction_source"", ""Start"", ""Stop""...",9d5b32b8ebd8d1ae,success,None



── dim_assembly_metadata ──


,Assembly_Accession,Assembly_Name,Organism_Name,Species_TaxID,Strain,Assembly_Level,RefSeq_Category,BioSample,BioProject,FTP_Path,Submission_Date,Is_Latest,Quality_Score,Is_RefSeq,Download_Status,Download_Date,Metadata_Only
0,GCF_055959565.1,ASM5595956v1,Xylella fastidiosa,2371,-,Complete Genome,na,SAMN46769223,-,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/055/959/565/GCF_055959565.1_ASM55...,2026/03/18 00:00,True,1010,True,success,2026-07-04,False
1,GCF_055583525.1,ASM5558352v1,Corynebacterium xerosis,1725,-,Scaffold,na,SAMN55302120,-,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/055/583/525/GCF_055583525.1_ASM55...,2026/03/01 00:00,True,110,True,success,2026-07-04,False
2,GCF_058500565.1,ASM5850056v1,Salmonella enterica,28901,-,Complete Genome,na,SAMN43257597,-,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/058/500/565/GCF_058500565.1_ASM58...,2026/06/29 00:00,True,1010,True,success,2026-07-04,False
3,GCF_058449145.1,ASM5844914v1,Bacillus cereus,1396,-,Complete Genome,na,SAMN47628865,-,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/058/449/145/GCF_058449145.1_ASM58...,2026/06/28 00:00,True,1010,True,success,2026-07-04,False
4,GCF_051904065.1,ASM5190406v1,Parabacteroides distasonis,823,-,Complete Genome,na,SAMN40870503,-,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/051/904/065/GCF_051904065.1_ASM51...,2025/08/10 00:00,True,1010,True,success,2026-07-04,False



── pipeline_run_provenance ──


,pipeline_run_timestamp,provider_name,provider_release,provider_snapshot_date,provider_schema_profile,provider_api_base_url,provider_provenance_mode,pbi_version,git_commit,download_records_count
0,2026-07-04T15:56:50Z,PhageScope,rolling,2026-05-11,phagescope_v1,https://phageapi.deepomics.org,config_pinned,None,None,229


2026-07-06 06:39:47,713 - INFO - 🔒 Database connection closed


---
## 7. Practical Reproducibility Checklist

Use this checklist when sharing or archiving a PBI-derived dataset or analysis.

### Data identity
- [ ] Record `pbi.__version__` in your analysis script / notebook header
- [ ] Record the `schema_profile` from the provenance JSON (`phagescope_v1`, etc.)
- [ ] Record the `snapshot_date` from the provenance JSON
- [ ] Archive `pipeline_run_provenance.json` alongside your results

### Code identity
- [ ] Tag or note the Git commit SHA used to run the pipeline
- [ ] Pin exact dependency versions (`pip freeze > requirements_lock.txt`)

### Data artefacts
- [ ] Include the DuckDB file checksum (or Zenodo DOI) in methods
- [ ] Archive the `public_data_manifest.json` for input traceability

### Quoting in publications

A minimal, reproducible reference sentence:

> Analysis used PBI v{pbi.__version__} (schema profile: `phagescope_v1`, PhageScope
> snapshot {snapshot_date}).  Source code and database available at
> https://github.com/ThibaultSchowing/PBI.

Substitute `{pbi.__version__}` and `{snapshot_date}` from your provenance artefact.


In [7]:
# Print a ready-to-use citation line from the current provenance
snapshot_date = provenance.get('snapshot_date', '<snapshot_date>') if provenance else '<snapshot_date>'
schema_profile = provenance.get('provider_schema_profile', 'phagescope_v1') if provenance else 'phagescope_v1'

print('Suggested citation line:')
print()
print(f'  PBI v{pbi.__version__} (schema profile: {schema_profile!r},'
      f' PhageScope snapshot {snapshot_date}).')
print('  https://github.com/ThibaultSchowing/PBI')


Suggested citation line:

  PBI v0.3.0 (schema profile: 'phagescope_v1', PhageScope snapshot <snapshot_date>).
  https://github.com/ThibaultSchowing/PBI


2026-07-06 06:39:59,106 - INFO -    ✅ Phage FASTA loaded in 12.09s (1,327,915 sequences)
2026-07-06 06:39:59,108 - INFO - 🔄 [Background] Loading protein FASTA: /data/processed/sequences/all_proteins.fasta
2026-07-06 06:55:47,644 - INFO -    ✅ Protein FASTA loaded in 948.54s (71,969,191 sequences)
2026-07-06 06:55:47,647 - INFO -    ℹ️  Using on-demand loading for 5,517 individual host files
2026-07-06 06:55:47,648 - INFO - 🎉 All FASTA files loaded in 960.63s
2026-07-06 06:56:13,412 - INFO - 🔍 Sample phage keys:
2026-07-06 06:56:13,413 - INFO -    - 'AE002163.1...'
2026-07-06 06:56:13,414 - INFO -    - 'AF009630.1...'
2026-07-06 06:56:13,415 - INFO -    - 'AF011378.1...'
2026-07-06 06:56:13,415 - INFO - 🔍 Sample protein keys:
2026-07-06 06:56:13,416 - INFO -    - 'AE002163.1 AAF39720.1...'
2026-07-06 06:56:13,417 - INFO -    - 'AE002163.1 AAF39721.1...'
2026-07-06 06:56:13,418 - INFO -    - 'AE002163.1 AAF39722.1...'


---
## Summary

| Layer | Artefact | Stable across |
|-------|----------|---------------|
| Package version | `pbi.__version__` in code | software releases |
| Schema profile | `public_data_provider.schema_profile` in config + DB | data schema versions |
| Run provenance | `pipeline_run_provenance.json` + DB tables | individual pipeline runs |

These three layers together give you exact, citable traceability from any analysis
result back to the specific software, data schema, and data snapshot that produced it.
